In [ ]:
# Cell 1: Imports, device, constants

import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder

# Device + shared constants used throughout training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 10
INPUT_SHAPE = (3, 180, 180)
BATCH_SIZE = 32
SEED = 42
LEARNING_RATE = 1e-3
DEBUG_EPOCHS = 1

print("Using device:", DEVICE)


In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder

# Paths
IMAGE_DIR = "Data/images_original"

# Hyperparameters
BATCH_SIZE = 32
SEED = 42

# Image preprocessing required by the coursework
image_transform = transforms.Compose([
    transforms.Resize((180, 180)),
    transforms.ToTensor()
])

# Load spectrogram image dataset
image_dataset = ImageFolder(
    root=IMAGE_DIR,
    transform=image_transform
)

# Check classes
print("Classes:", image_dataset.classes)
print("Class to index:", image_dataset.class_to_idx)
print("Total image samples:", len(image_dataset))

# Train / validation / test split: 70% / 20% / 10%
total_size = len(image_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    image_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Check one batch
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Example labels:", labels[:10])


In [ ]:
# Cell 3: Utility functions

def set_seed(seed=SEED):
    """Set random seeds for reproducibility across Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def count_parameters(model):
    """Return number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def smoke_test_model(model, input_shape=INPUT_SHAPE, batch_size=4, num_classes=NUM_CLASSES, device=DEVICE):
    """
    Quick shape/device sanity check for a model.
    Ensures output shape is [batch_size, num_classes].
    """
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        x = torch.randn(batch_size, *input_shape).to(device)
        y = model(x)

    assert y.shape == (batch_size, num_classes), (
        f"Smoke test failed: expected {(batch_size, num_classes)}, got {tuple(y.shape)}"
    )
    print(f"Smoke test passed. Output shape: {tuple(y.shape)}")
    print(f"Trainable parameters: {count_parameters(model):,}")


In [ ]:
# Cell 4: Training and evaluation functions

criterion = nn.CrossEntropyLoss()
results_records = []


def train_one_epoch(model, loader, optimizer, criterion, device=DEVICE):
    """Train for one epoch and return average training loss."""
    model.train()
    running_loss = 0.0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total += images.size(0)

    return running_loss / total


def evaluate(model, loader, criterion, device=DEVICE):
    """Evaluate model and return (avg_loss, accuracy_percent)."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def train_model(model, model_name, architecture, optimizer_name, epochs, train_loader, val_loader, criterion, lr=LEARNING_RATE, device=DEVICE):
    """
    Full training loop with validation tracking.
    Returns model loaded with best validation checkpoint and history dictionary.
    """
    set_seed(SEED)
    model = model.to(device)

    if optimizer_name.lower() == "rmsprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    elif optimizer_name.lower() == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"[{model_name}] Epoch {epoch}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%"
        )

    model.load_state_dict(best_state)
    return model, history


def test_model(model, loader, criterion, device=DEVICE):
    """Evaluate trained model on test set."""
    test_loss, test_acc = evaluate(model, loader, criterion, device)
    return test_loss, test_acc


def append_result(model_name, architecture, optimizer_name, epochs, history, test_loss, test_acc):
    """Append one run result to shared records list."""
    results_records.append({
        "model_name": model_name,
        "architecture": architecture,
        "optimizer": optimizer_name,
        "epochs": epochs,
        "final_training_loss": history["train_loss"][-1],
        "final_validation_loss": history["val_loss"][-1],
        "validation_accuracy": history["val_acc"][-1],
        "test_loss": test_loss,
        "test_accuracy": test_acc,
    })


In [ ]:
# Cell 5: Net1 model definition only

class Net1(nn.Module):
    """Net1: fully connected network with exactly two hidden layers."""
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * 180 * 180, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


In [ ]:
# Cell 6: Net1 smoke test and optional 1-epoch debug run

net1 = Net1()
smoke_test_model(net1)

RUN_DEBUG_NET1 = False  # Set True to run 1 epoch quick debug training
if RUN_DEBUG_NET1:
    net1_debug = Net1()
    net1_debug, net1_debug_history = train_model(
        model=net1_debug,
        model_name="Net1",
        architecture="FC(Flatten->512->128->10)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
    )
    dbg_test_loss, dbg_test_acc = test_model(net1_debug, test_loader, criterion)
    print(f"Net1 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net1 debug run skipped. Set RUN_DEBUG_NET1=True to execute.")


In [ ]:
# Cell 7: Net1 50-epoch and 100-epoch training calls

RUN_FULL_NET1 = False  # Set True when you want full Net1 training
if RUN_FULL_NET1:
    for n_epochs in [50, 100]:
        net1_run = Net1()
        net1_run, net1_history = train_model(
            model=net1_run,
            model_name="Net1",
            architecture="FC(Flatten->512->128->10)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
        )
        net1_test_loss, net1_test_acc = test_model(net1_run, test_loader, criterion)
        append_result("Net1", "FC(Flatten->512->128->10)", "Adam", n_epochs, net1_history, net1_test_loss, net1_test_acc)
        print(f"Net1 ({n_epochs} epochs) test acc: {net1_test_acc:.2f}%")
else:
    print("Net1 full training skipped. Set RUN_FULL_NET1=True to execute.")


In [ ]:
# Cell 8: Net2 model definition only

class Net2(nn.Module):
    """
    Net2: Coursework Figure 1 style CNN
    Input -> Conv+ReLU -> Conv+ReLU -> MaxPool
          -> Conv+ReLU -> Conv+ReLU -> MaxPool
          -> FC+ReLU -> FC output
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, *INPUT_SHAPE)
            flattened_dim = self.features(dummy).view(1, -1).size(1)

        self.classifier = nn.Sequential(
            nn.Linear(flattened_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [ ]:
# Cell 9: Net2 smoke test and optional 1-epoch debug run

net2 = Net2()
smoke_test_model(net2)

RUN_DEBUG_NET2 = False
if RUN_DEBUG_NET2:
    net2_debug = Net2()
    net2_debug, net2_debug_history = train_model(
        model=net2_debug,
        model_name="Net2",
        architecture="CNN(Fig1 4conv+2pool+fc256)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
    )
    dbg_test_loss, dbg_test_acc = test_model(net2_debug, test_loader, criterion)
    print(f"Net2 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net2 debug run skipped. Set RUN_DEBUG_NET2=True to execute.")


In [ ]:
# Cell 10: Net2 50-epoch and 100-epoch training calls

RUN_FULL_NET2 = False
if RUN_FULL_NET2:
    for n_epochs in [50, 100]:
        net2_run = Net2()
        net2_run, net2_history = train_model(
            model=net2_run,
            model_name="Net2",
            architecture="CNN(Fig1 4conv+2pool+fc256)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
        )
        net2_test_loss, net2_test_acc = test_model(net2_run, test_loader, criterion)
        append_result("Net2", "CNN(Fig1 4conv+2pool+fc256)", "Adam", n_epochs, net2_history, net2_test_loss, net2_test_acc)
        print(f"Net2 ({n_epochs} epochs) test acc: {net2_test_acc:.2f}%")
else:
    print("Net2 full training skipped. Set RUN_FULL_NET2=True to execute.")


In [ ]:
# Cell 11: Net3 model definition only

class Net3(nn.Module):
    """
    Net3: Same architecture as Net2, but with BatchNorm2d
    after each convolution and before ReLU.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, *INPUT_SHAPE)
            flattened_dim = self.features(dummy).view(1, -1).size(1)

        self.classifier = nn.Sequential(
            nn.Linear(flattened_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [ ]:
# Cell 12: Net3 smoke test and optional 1-epoch debug run

net3 = Net3()
smoke_test_model(net3)

RUN_DEBUG_NET3 = False
if RUN_DEBUG_NET3:
    net3_debug = Net3()
    net3_debug, net3_debug_history = train_model(
        model=net3_debug,
        model_name="Net3",
        architecture="CNN+BN(Fig1 4conv+2pool+fc256)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
    )
    dbg_test_loss, dbg_test_acc = test_model(net3_debug, test_loader, criterion)
    print(f"Net3 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net3 debug run skipped. Set RUN_DEBUG_NET3=True to execute.")


In [ ]:
# Cell 13: Net3 50-epoch and 100-epoch training calls

RUN_FULL_NET3 = False
if RUN_FULL_NET3:
    for n_epochs in [50, 100]:
        net3_run = Net3()
        net3_run, net3_history = train_model(
            model=net3_run,
            model_name="Net3",
            architecture="CNN+BN(Fig1 4conv+2pool+fc256)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
        )
        net3_test_loss, net3_test_acc = test_model(net3_run, test_loader, criterion)
        append_result("Net3", "CNN+BN(Fig1 4conv+2pool+fc256)", "Adam", n_epochs, net3_history, net3_test_loss, net3_test_acc)
        print(f"Net3 ({n_epochs} epochs) test acc: {net3_test_acc:.2f}%")
else:
    print("Net3 full training skipped. Set RUN_FULL_NET3=True to execute.")


In [ ]:
# Cell 14: Net4 training setup only

# Net4 must use the SAME architecture as Net3.
# We reuse Net3 directly to keep comparison fair.
Net4 = Net3
NET4_ARCH = "CNN+BN(Fig1 4conv+2pool+fc256)"
NET4_OPTIMIZER = "RMSprop"

print("Net4 setup complete: using Net3 architecture with RMSprop optimizer.")


In [ ]:
# Cell 15: Net4 smoke test and optional 1-epoch debug run

net4 = Net4()
smoke_test_model(net4)

RUN_DEBUG_NET4 = False
if RUN_DEBUG_NET4:
    net4_debug = Net4()
    net4_debug, net4_debug_history = train_model(
        model=net4_debug,
        model_name="Net4",
        architecture=NET4_ARCH,
        optimizer_name=NET4_OPTIMIZER,
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
    )
    dbg_test_loss, dbg_test_acc = test_model(net4_debug, test_loader, criterion)
    print(f"Net4 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net4 debug run skipped. Set RUN_DEBUG_NET4=True to execute.")


In [ ]:
# Cell 16: Net4 50-epoch and 100-epoch training calls

RUN_FULL_NET4 = False
if RUN_FULL_NET4:
    for n_epochs in [50, 100]:
        net4_run = Net4()
        net4_run, net4_history = train_model(
            model=net4_run,
            model_name="Net4",
            architecture=NET4_ARCH,
            optimizer_name=NET4_OPTIMIZER,
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
        )
        net4_test_loss, net4_test_acc = test_model(net4_run, test_loader, criterion)
        append_result("Net4", NET4_ARCH, NET4_OPTIMIZER, n_epochs, net4_history, net4_test_loss, net4_test_acc)
        print(f"Net4 ({n_epochs} epochs) test acc: {net4_test_acc:.2f}%")
else:
    print("Net4 full training skipped. Set RUN_FULL_NET4=True to execute.")


In [ ]:
# Cell 17: Results table and CSV saving

# Build DataFrame from completed training runs.
results_df = pd.DataFrame(results_records)

if results_df.empty:
    print("No full training runs have been logged yet.")
    print("Run Net1-Net4 full training cells with RUN_FULL_* = True, then rerun this cell.")
else:
    results_df = results_df.sort_values(["model_name", "epochs"]).reset_index(drop=True)
    display(results_df)

# Save CSV even if empty, so coursework file path is always created.
results_df.to_csv("results_image_models.csv", index=False)
print("Saved results to results_image_models.csv")
